# III. Comparaison des tryptiques.csv

## 1) Imports et chemins

In [12]:
import pandas as pd
import numpy as np

TRIPTYQUES_FOLDER = "../results/csv_triptyques/"

auto = pd.read_csv(f"{TRIPTYQUES_FOLDER}triptyques_tdm.csv")
manuel = pd.read_csv(F"{TRIPTYQUES_FOLDER}triptyques_chap1to5_sacr.csv")

## 2) Est ce que les deux fichier on les memes colonnes ?

In [13]:

print("Colonnes automatique :")
print(auto.columns.tolist())

print("\nColonnes manuel :")
print(manuel.columns.tolist())


Colonnes automatique :
['Phrase', 'Sujet', 'Verbe', 'Objet', 'Dep_sujet', 'Dep_verbe', 'Dep_objet', 'ID_sujet', 'ID_verbe', 'ID_objet', 'IDs_sujet', 'IDs_objet', 'Lemme_verbe', 'Head_verbe', 'Annotation_verbe_col8', 'Annotation_verbe_col9', 'Num_phrase', 'Num_paragr', 'Sentence_index', 'personnages_sujet', 'personnages_objet', 'personnages_triptyque', 'lieux_sujet', 'lieux_objet', 'lieux_triptyque', 'types_lieux_triptyque', 'a_personnage_triptyque', 'a_lieu_triptyque', 'a_personnage_et_lieu_triptyque']

Colonnes manuel :
['Phrase', 'Sujet', 'Verbe', 'Objet', 'Dep_sujet', 'Dep_verbe', 'Dep_objet', 'ID_sujet', 'ID_verbe', 'ID_objet', 'IDs_sujet', 'IDs_objet', 'Lemme_verbe', 'Head_verbe', 'Annotation_verbe_col8', 'Annotation_verbe_col9', 'Num_phrase', 'Num_paragr', 'Sentence_index', 'personnages_sujet', 'personnages_objet', 'personnages_triptyque', 'lieux_sujet', 'lieux_objet', 'lieux_triptyque', 'types_lieux_triptyque', 'a_personnage_triptyque', 'a_lieu_triptyque', 'a_personnage_et_lieu_

## 3) Filter csv automatiques par rapport aux chapitres annotés manuellement (1 à 5 pour l'instant) 

In [14]:
phrases_manuelles = set(manuel["Phrase"].dropna())

auto_chap1to5 = auto[auto["Phrase"].isin(phrases_manuelles)].copy()

print("Triptyques automatiques avant filtre :", len(auto))
print("Triptyques automatiques après filtre chap. 1 à 5 :", len(auto_chap1to5))
print("Triptyques manuels chap. 1 à 5 :", len(manuel))

Triptyques automatiques avant filtre : 13047
Triptyques automatiques après filtre chap. 1 à 5 : 207
Triptyques manuels chap. 1 à 5 : 1255


## 4) Création d'une clé pour chaque tryptique

In [21]:
key_cols = ["ID_sujet", "ID_verbe", "ID_objet"]

for df in [auto_chap1to5, manuel]:
    df["cle_triptyque"] = (
        df[key_cols]
        .fillna("")
        .astype(str)
        .apply(lambda row: " || ".join(row), axis=1)
    )

## 5) Comparaison des clés

In [22]:
cles_auto = set(auto_chap1to5["cle_triptyque"])
cles_manuel = set(manuel["cle_triptyque"])

communs = cles_auto & cles_manuel
seulement_auto = cles_auto - cles_manuel
seulement_manuel = cles_manuel - cles_auto

print("Triptyques communs :", len(communs))
print("Triptyques seulement automatiques :", len(seulement_auto))
print("Triptyques seulement manuels :", len(seulement_manuel))

Triptyques communs : 164
Triptyques seulement automatiques : 1
Triptyques seulement manuels : 730


## 6) Quelles sont les dépéndances ou il y a le plus de différences ?

In [ ]:
# On récupère les triptyques produits uniquement par l'annotation automatique.
df_seulement_auto = auto_chap1to5[auto_chap1to5["cle_triptyque"].isin(seulement_auto)].copy()
df_seulement_auto["source_comparaison"] = "seulement_automatique"

# On récupère les triptyques présents uniquement dans l'annotation manuelle.
df_seulement_manuel = manuel[manuel["cle_triptyque"].isin(seulement_manuel)].copy()
df_seulement_manuel["source_comparaison"] = "seulement_manuel"

# On rassemble les deux types de différences dans un seul tableau.
differences_triptyques = pd.concat(
    [df_seulement_auto, df_seulement_manuel],
    ignore_index=True
)

# On compte les différences par combinaison de dépendances :
# dépendance du sujet, dépendance du verbe, dépendance de l'objet.
resume_dependances = (
    differences_triptyques
    .groupby(["source_comparaison", "Dep_sujet", "Dep_verbe", "Dep_objet"])
    .size()
    .reset_index(name="nombre")
    .sort_values("nombre", ascending=False)
)

resume_dependances

,source_comparaison,Dep_sujet,Dep_verbe,Dep_objet,nombre
38,seulement_manuel,nsubj,root,obl:mod,67
36,seulement_manuel,nsubj,root,obj,48
8,seulement_manuel,nsubj,acl:relcl,obj,42
22,seulement_manuel,nsubj,conj,obj,34
41,seulement_manuel,nsubj,xcomp,obj,31
10,seulement_manuel,nsubj,acl:relcl,obl:mod,27
13,seulement_manuel,nsubj,advcl,obj,26
37,seulement_manuel,nsubj,root,obl:arg,25
23,seulement_manuel,nsubj,conj,obl:arg,16
34,seulement_manuel,nsubj,root,expl:comp,16
